# Permian flare sites — maps, figures, and animations



| need | file | why this one |
|---|---|---|
| site geometry | `upstream.shp` | clean polygons; the CSV's `geometry` column was truncated to 254 chars by a DBF round-trip |
| site attributes | `permian_sites_full.csv` | already the Permian subset, with `state` and `ID2015…ID2021` |
| monthly signal | `vnf_sites_aggregated/*.csv` | site × month sums and counts; means are computed here, not upstream |

Join key is the **string** `Catalog ID` (Permian) ↔ `id` (upstream), *not* the integer `ID`
(which is just `upstream`'s positional index).

**Products**

1. `permian_sites_map.html` — folium, per-state toggles, footprint polygons, temporal view
2. `permian_sites_all.png` / `permian_sites_tx_only.png` — static figures for the writeup
3. `monthly_flaring_log_dark.html` — log-scaled radiant heat on a dark basemap
4. `monthly_flaring_points.html` — linear scale, size + color, light basemap
5. `monthly_flaring_density.html` — smoothed density (impressionistic; see caveats)
6. `monthly_flaring_{points,density}.{mp4,gif}` — the same two animations, for slides

**Two conventions that hold throughout, and are not negotiable:**

- The color scale and the map extent are **fixed across every frame** of an animation. If they float,
  months aren't comparable and the animation is decoration rather than evidence.
- Cloud-only months are `NaN`, never `0`. "Not observed" and "not flaring" are different facts, and
  conflating them manufactures exactly the decoupling this project is trying to detect.

## Setup

In [1]:
# !pip install pyshp folium pandas numpy plotly matplotlib scipy imageio imageio-ffmpeg pillow
import shapefile          # pyshp
import pandas as pd
import numpy as np
import folium
from pathlib import Path

PERMIAN  = Path("../../../data/processed/nightfire/permian_sites_full.csv")
SHP      = Path("../../../data/processed/nightfire/VNF_multiyear_by_type_2012-2021_v20220822/upstream.shp")
AGG_DIR  = Path("../../../data/processed/nightfire/vnf_sites_aggregated/")
ENCODING = "ISO-8859-1"                     # from upstream.cpg

OUTDIR = Path("../../../visualizations/nightfire")
OUTDIR.mkdir(parents=True, exist_ok=True)

STATE_COL = {"NM": "#d62728", "TX": "#1f77b4"}
DROP_NM = True        # True -> Texas-only everywhere except Product 2 (which always shows both)

## Geometry from the shapefile

Build an `id -> rings` lookup. Coordinates are stored GeoJSON-style `[lon, lat]`.

In [2]:
sf = shapefile.Reader(str(SHP), encoding=ENCODING)
fields = [f[0] for f in sf.fields[1:]]
id_pos = fields.index("id")

geom = {}
for shp, rec in zip(sf.shapes(), sf.records()):
    pts_xy = shp.points
    parts  = list(shp.parts) + [len(pts_xy)]
    rings  = [[[lon, lat] for lon, lat in pts_xy[parts[i]:parts[i+1]]]
              for i in range(len(parts) - 1)]
    geom[rec[id_pos]] = rings

print(f"{len(geom)} polygons read from {SHP.name}")

13446 polygons read from upstream.shp


## Attributes from the Permian CSV, joined to geometry

In [3]:
p = pd.read_csv(PERMIAN)
p["rings"] = p["Catalog ID"].map(geom)

matched = p["rings"].notna().sum()
print(f"{matched} / {len(p)} Permian sites matched to a polygon")
print("unmatched (absent from upstream in every format):")
print(p.loc[p["rings"].isna(), ["Catalog ID", "ID", "state"]].to_string(index=False))

2323 / 2328 Permian sites matched to a polygon
unmatched (absent from upstream in every format):
                                            Catalog ID   ID state
 IR_2012-2020_tile_-105,30,-90,45_feature_315_source_0 7413    NM
 IR_2012-2020_tile_-105,30,-90,45_feature_692_source_0 7182    TX
 IR_2012-2020_tile_-105,30,-90,45_feature_488_source_0 8820    TX
IR_2012-2020_tile_-105,30,-90,45_feature_1844_source_0 9162    TX
IR_2012-2020_tile_-105,30,-90,45_feature_1555_source_3 9503    TX


## Temporal feature: last year detected

From the annual catalog columns `ID2015…ID2021` (non-null = present that year).

**Caveat that has to travel with this feature.** The annual columns stop at 2021 and reflect the
2012–2020 catalog vintage, so "last seen 2021" means *still present at series end*, not *survived the
NM rule*. The catalog structurally cannot show post-2021 disappearance. Descriptive only — this is not
the decoupling test, and it should not be presented as if it were.

In [4]:
ycols = [c for c in p.columns if c.startswith("ID20")]
yrs   = [int(c[2:]) for c in ycols]

def last_year(row):
    present = [y for y, c in zip(yrs, ycols) if pd.notna(row[c])]
    return max(present) if present else np.nan

p["last_year"] = p.apply(last_year, axis=1)
print(p["last_year"].value_counts(dropna=False).sort_index())
# sites with no annual ID populated -> NaN -> drawn grey below

last_year
2015.0      39
2016.0      31
2017.0      95
2018.0     307
2019.0       1
2020.0     468
2021.0    1352
NaN         35
Name: count, dtype: int64


---
## Product 1 — folium map

Three views, each split into a **New Mexico group and a Texas group**, so the layer control gives you a
per-state checkbox: unchecking `NM` leaves a clean Texas-only map in every view. The old
`state × in_delaware` four-way palette is gone — the Delaware sub-basin subclassification was doing no
work in this figure and cost half the color budget.

- **points by state** (on by default) — size ∝ `RH_mean`
- **footprint polygons** (off) — real geometry, ~280 m across, only legible when zoomed right in
- **last year detected** (off) — the temporal view, with the caveat above

In [5]:
pts = p.dropna(subset=["Latitude", "Longitude"]).copy()

fmap = folium.Map(location=[pts["Latitude"].mean(), pts["Longitude"].mean()],
                  zoom_start=7, tiles="CartoDB positron")

states = [s for s in ["NM", "TX"] if s in set(pts["state"])]

rh = pts["RH_mean"].clip(lower=0).fillna(0)
pts["radius"] = 2 + 6 * np.sqrt(rh / (rh.max() or 1))

# ---- Layer 1: points by state, one FeatureGroup per state -> one checkbox per state ----
for st in states:
    g = folium.FeatureGroup(name=f"points: {st}", show=True)
    for _, r in pts[pts["state"] == st].iterrows():
        col = STATE_COL[st]
        folium.CircleMarker(
            [r["Latitude"], r["Longitude"]], radius=float(r["radius"]),
            color=col, weight=0.5, fill=True, fill_color=col, fill_opacity=0.6,
            popup=folium.Popup(f"ID {r['ID']}<br>{st}<br>"
                               f"RH_mean {r['RH_mean']:.3f}<br>last seen {r['last_year']}",
                               max_width=200),
        ).add_to(g)
    g.add_to(fmap)

# ---- Layer 2: footprint polygons, per state, off by default ----
for st in states:
    feats = [{"type": "Feature",
              "properties": {"state": st, "id": int(r["ID"])},
              "geometry": {"type": "Polygon", "coordinates": r["rings"]}}
             for _, r in pts[pts["state"] == st].iterrows()
             if isinstance(r["rings"], list)]
    folium.GeoJson(
        {"type": "FeatureCollection", "features": feats},
        name=f"footprints: {st}", show=False,
        style_function=lambda x, c=STATE_COL[st]: {"color": c, "weight": 1, "fillOpacity": 0.4},
        tooltip=folium.GeoJsonTooltip(fields=["id", "state"]),
    ).add_to(fmap)

# ---- Layer 3: points by last year detected, per state, off by default ----
year_col = {2015: "#440154", 2016: "#46327e", 2017: "#365c8d", 2018: "#277f8e",
            2019: "#1fa187", 2020: "#4ac16d", 2021: "#fde725"}
for st in states:
    g = folium.FeatureGroup(name=f"last year detected: {st}", show=False)
    for _, r in pts[pts["state"] == st].iterrows():
        col = year_col.get(r["last_year"], "#999999")
        folium.CircleMarker(
            [r["Latitude"], r["Longitude"]], radius=3,
            color=col, weight=0.5, fill=True, fill_color=col, fill_opacity=0.7,
            popup=f"ID {r['ID']} | {st} | last seen {r['last_year']}",
        ).add_to(g)
    g.add_to(fmap)

folium.LayerControl(collapsed=False).add_to(fmap)

legend = '''
<div style="position: fixed; bottom: 24px; left: 24px; z-index: 9999; background: white;
            padding: 10px 12px; border: 1px solid #999; border-radius: 4px;
            font: 12px sans-serif; line-height: 1.5;">
<b>flare sites by state</b><br>
<span style="color:#d62728;">&#9679;</span> New Mexico
&nbsp; <span style="color:#1f77b4;">&#9679;</span> Texas<br>
<span style="color:#555;">size &prop; RH_mean &middot; uncheck a state in the control box</span>
</div>'''
fmap.get_root().html.add_child(folium.Element(legend))

OUT = OUTDIR / "permian_sites_map.html"
fmap.save(str(OUT))
print(f"{len(pts)} sites plotted {pts['state'].value_counts().to_dict()} -> {OUT}")
# no bare `fmap` here — that line is what embeds the whole map in the .ipynb

2328 sites plotted {'TX': 1769, 'NM': 559} -> ../../../visualizations/nightfire/permian_sites_map.html


---
## Product 2 — static figures, with and without New Mexico

For the writeup and the slides. Two deliberate choices:

- **The axis limits are locked to the full (NM + TX) extent in both figures.** If the Texas-only panel
  is allowed to autoscale it silently zooms, and the two figures stop being the same picture minus a
  set of points — which is the entire thing they're meant to show.
- **The state line is drawn from its actual definition**, not from a shapefile: the NM/TX boundary runs
  south along the 103°W meridian to the 32°N parallel, then west along 32°N. So in this window,
  New Mexico is exactly `lon < -103 and lat > 32`.

That second point buys a free QC check, run at the bottom of the cell. The `state` column is the
treatment assignment for the whole DiD; if any site is mislabelled it will be mislabelled *at the
border*, which is precisely where identification lives. Worth confirming once that the count is zero.

In [6]:
import matplotlib.pyplot as plt

PAD  = 0.15
XLIM = (pts["Longitude"].min() - PAD, pts["Longitude"].max() + PAD)
YLIM = (pts["Latitude"].min()  - PAD, pts["Latitude"].max()  + PAD)
RHMAX = float(pts["RH_mean"].clip(lower=0).fillna(0).max()) or 1.0

def plot_sites(df, title, path):
    fig, ax = plt.subplots(figsize=(9, 8), dpi=220)
    for st in ["TX", "NM"]:
        sub = df[df["state"] == st]
        if sub.empty:
            continue
        s = 3 + 55 * np.sqrt(sub["RH_mean"].clip(lower=0).fillna(0) / RHMAX)
        ax.scatter(sub["Longitude"], sub["Latitude"], s=s, c=STATE_COL[st],
                   alpha=0.55, linewidths=0, label=f"{st} (n={len(sub)})")
    # NM/TX border: south along 103W to 32N, then west along 32N
    ax.plot([-103, -103], [32, YLIM[1]], color="0.35", lw=1.0, ls="--", zorder=0)
    ax.plot([XLIM[0], -103], [32, 32],  color="0.35", lw=1.0, ls="--", zorder=0)
    ax.set_xlim(*XLIM); ax.set_ylim(*YLIM)
    ax.set_aspect(1 / np.cos(np.deg2rad(np.mean(YLIM))))   # degrees -> true ground scale
    ax.set_xlabel("longitude"); ax.set_ylabel("latitude")
    ax.set_title(title, loc="left")
    ax.legend(loc="upper right", frameon=False, markerscale=1.5)
    ax.grid(alpha=0.15, lw=0.5)
    for sp in ax.spines.values():
        sp.set_visible(False)
    fig.savefig(path, bbox_inches="tight")
    plt.close(fig)
    print("wrote", path)

plot_sites(pts, r"Permian VIIRS flare sites (size $\propto$ mean radiant heat)",
           OUTDIR / "permian_sites_all.png")
plot_sites(pts[pts["state"] == "TX"], "Permian VIIRS flare sites — Texas only",
           OUTDIR / "permian_sites_tx_only.png")

bad = pts[(pts["state"] == "NM") & ((pts["Longitude"] > -103) | (pts["Latitude"] < 32))]
print(f"QC — NM-labelled sites falling outside NM by the 103W/32N border: {len(bad)}")

wrote ../../../visualizations/nightfire/permian_sites_all.png
wrote ../../../visualizations/nightfire/permian_sites_tx_only.png
QC — NM-labelled sites falling outside NM by the 103W/32N border: 0


---
## The monthly panel

One long table: every site × every month. Three things happen here and each is a decision, not a
formality.

**The metric is radiant heat per clear look**, `rh_sum / (n_detect + n_nondet_cm0 + n_nondet_cm1)`.
A clear look is a detection *or* a clear-sky non-detection — i.e. an opportunity to see a flare, taken.
Dividing by opportunities removes the observation-frequency confound; a month with only cloudy looks
carries no information and becomes `NaN`, not zero.

**Site coordinates are pinned to the site's median lat/lon.** The reported position jitters a little
month to month with the overpass geometry; unpinned, sites visibly wobble across frames in the
animation and it reads as movement of the flares themselves.

**State is assigned from the border geometry** (`lon < -103 and lat > 32` ⇒ NM), because the monthly
aggregates carry no `state` column. This is what lets the animation show the two arms of the experiment
rather than an undifferentiated basin.

In [7]:
mon = pd.concat([pd.read_csv(f) for f in sorted(AGG_DIR.glob("*.csv"))],
                ignore_index=True)

mon["n_clear_look"] = mon["n_detect"] + mon["n_nondet_cm0"] + mon["n_nondet_cm1"]
mon["rh_per_clear_look"] = np.where(
    mon["n_clear_look"] > 0, mon["rh_sum"] / mon["n_clear_look"], np.nan)

xy = mon.groupby("flare_id")[["lat", "lon"]].median()
mon["lat"] = mon["flare_id"].map(xy["lat"])
mon["lon"] = mon["flare_id"].map(xy["lon"])

mon["state"] = np.where((mon["lon"] < -103) & (mon["lat"] > 32), "NM", "TX")

print(f"{len(mon):,} site-months | {mon.flare_id.nunique():,} sites | "
      f"{mon.year_month.nunique()} months "
      f"({mon.year_month.min()} -> {mon.year_month.max()})")
print(mon.groupby("state")["flare_id"].nunique())
print(f"only-cloudy site-months (NaN, not zero): {mon['rh_per_clear_look'].isna().sum():,}")

796,308 site-months | 2,324 sites | 173 months (2012-03 -> 2026-07)
state
NM     558
TX    1766
Name: flare_id, dtype: int64
only-cloudy site-months (NaN, not zero): 580


In [8]:
if DROP_NM:
    n0_sites, n0_rows = len(pts), len(mon)
    pts = pts[pts["state"] == "TX"].copy()
    mon = mon[mon["state"] == "TX"].copy()
    print(f"DROP_NM: sites {n0_sites} -> {len(pts)} | site-months {n0_rows:,} -> {len(mon):,}")
else:
    print("DROP_NM is False — New Mexico retained in all products")

DROP_NM: sites 2328 -> 1769 | site-months 796,308 -> 605,112


---
## Product 3 — log-scaled radiant heat, dark basemap

This replaces the old Inferno-on-`carto-positron` version, whose color scale was not merely ugly but
**inverted in effect**. Two compounding problems:

1. Inferno's low end is near-black. On a white basemap that makes the *quietest* sites the most
   visually salient objects on the map, while the actual flares — mid-scale orange — recede.
2. `rh_per_clear_look` is heavy-tailed. A linear 0→p95 range crushes almost every site into the bottom
   decile of the ramp, so the scale is spending its whole dynamic range on the handful of sites you
   could already find by eye.

Fixed by logging the color and putting the fire colormap on a dark basemap, where it belongs. The
colorbar ticks are relabelled back into MW so the axis stays readable.

**The bottom of the scale is a floor, not a zero.** Anything quieter than `FLOOR` MW is clamped there.
Say so in any caption — otherwise the floor reads as a measurement.

In [9]:
import plotly.express as px

plot = mon.dropna(subset=["rh_per_clear_look"]).sort_values("year_month").copy()

FLOOR = 0.01                                   # MW
plot["log_rh"] = np.log10(plot["rh_per_clear_look"].clip(lower=FLOOR))
lo = np.log10(FLOOR)
hi = float(np.log10(max(plot["rh_per_clear_look"].quantile(0.995), FLOOR * 10)))
ticks  = np.arange(np.ceil(lo), np.floor(hi) + 1)
months = sorted(plot["year_month"].unique())

fig = px.scatter_map(          # plotly < 5.24: px.scatter_mapbox + mapbox_style=...
    plot, lat="lat", lon="lon",
    color="log_rh", range_color=(lo, hi),
    color_continuous_scale="Inferno",
    animation_frame="year_month", category_orders={"year_month": months},
    hover_name="flare_id",
    hover_data={"rh_per_clear_look": ":.3f", "n_clear_look": True,
                "state": True, "lat": False, "lon": False, "log_rh": False},
    zoom=6, height=650,
)
fig.update_layout(map_style="carto-darkmatter", margin=dict(l=0, r=0, t=30, b=0))
fig.update_traces(marker={"size": 7, "opacity": 0.9})
fig.update_coloraxes(colorbar=dict(
    title="RH / clear look<br>(MW, log)",
    tickvals=ticks, ticktext=[f"{10**t:g}" for t in ticks]))
try:
    fig.layout.updatemenus[0].buttons[0].args[1]["frame"]["duration"] = 400
except Exception:
    pass

OUT = OUTDIR / "monthly_flaring_log_dark.html"
fig.write_html(str(OUT), include_plotlyjs="cdn")
print("->", OUT)

-> ../../../visualizations/nightfire/monthly_flaring_log_dark.html


---
## Product 4 — points slider, linear scale

The version that already worked. Zeros recede in **both** channels — pale on a pale-to-dark ramp, and
small under a `sqrt` size map — so a quiet site is quiet in every visual dimension rather than merely
one. Kept alongside Product 3 because the two answer different questions: this one shows *where the big
flares are*, the log version shows *the whole population, including the tail of small ones*.

In [10]:
plot = mon.dropna(subset=["rh_per_clear_look"]).sort_values("year_month").copy()
plot["mksize"] = np.sqrt(plot["rh_per_clear_look"])       # 0 -> ~invisible, high -> big
vmax   = float(plot["rh_per_clear_look"].quantile(0.95))  # fixed across frames
months = sorted(plot["year_month"].unique())

fig = px.scatter_map(
    plot, lat="lat", lon="lon",
    color="rh_per_clear_look", range_color=(0, vmax),
    color_continuous_scale="YlOrRd",
    size="mksize", size_max=14,
    animation_frame="year_month", category_orders={"year_month": months},
    hover_name="flare_id", zoom=6, height=650,
    labels={"rh_per_clear_look": "mean RH / clear look (MW)"},
)
fig.update_layout(map_style="carto-positron", margin=dict(l=0, r=0, t=30, b=0))
fig.update_traces(marker={"opacity": 0.85})
try:
    fig.layout.updatemenus[0].buttons[0].args[1]["frame"]["duration"] = 400
except Exception:
    pass

OUT = OUTDIR / "monthly_flaring_points.html"
fig.write_html(str(OUT), include_plotlyjs="cdn")
print("->", OUT)

# quiet sites GONE rather than tiny?      plot = plot[plot.rh_per_clear_look > 0]
# faintly visible ("present, not flaring")? plot["mksize"] = np.sqrt(plot.rh_per_clear_look) + 0.15

-> ../../../visualizations/nightfire/monthly_flaring_points.html


---
## Product 5 — smoothed density

Impressionistic, and honest about it. `radius=25` is a **pixel** knob, not a physical one: the apparent
size of a hot region changes with zoom level and has no interpretation in kilometres. Fine as a "where
is the basin lighting up" visual, useless as a measurement. Do not put a number on anything read off
this map.

In [11]:
plot  = mon.dropna(subset=["rh_per_clear_look"]).sort_values("year_month").copy()
months = sorted(plot["year_month"].unique())
cmax   = float(plot["rh_per_clear_look"].quantile(0.98))

fig = px.density_map(
    plot, lat="lat", lon="lon", z="rh_per_clear_look",
    radius=25,                                    # <-- pixels, NOT metres
    range_color=(0, cmax), color_continuous_scale="Inferno",
    animation_frame="year_month", category_orders={"year_month": months},
    zoom=6, height=650,
)
fig.update_layout(map_style="carto-positron", margin=dict(l=0, r=0, t=30, b=0))

OUT = OUTDIR / "monthly_flaring_density.html"
fig.write_html(str(OUT), include_plotlyjs="cdn")
print("->", OUT)

-> ../../../visualizations/nightfire/monthly_flaring_density.html


---
## Product 6 — mp4 + gif for slides

The html above is the zoomable artifact; this is the one you play in a talk. Frames are rendered in
**matplotlib, not plotly**, for three reasons: kaleido's static export of maplibre figures is fragile
and wants tiles at render time; frame-by-frame control is needed for the outage banner below; and the
density panel gets an explicit, statable kernel (a 2-D histogram plus a Gaussian filter with a stated
`sigma`) instead of plotly's non-physical pixel radius.

**The August 2022 SNPP outage is handled here, visibly.** Clear looks collapse to a few percent of
normal that month. `rh_per_clear_look` doesn't go to zero — it goes to *noise* — and on a 170-frame
animation that renders as a dramatic basin-wide event that never happened. Someone in the audience will
ask about it. The cell detects the outage from a 20% threshold on a rolling median of basin-wide clear
looks and renders those months as an explicit banner frame. Silently dropping them would be worse: the
timeline would still advance, and the video would quietly assert that nothing unusual occurred.

Fixed extent, fixed color limits, and a fixed color scale computed from the **non-outage** months only.

In [12]:
# !pip install imageio imageio-ffmpeg pillow
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from scipy.ndimage import gaussian_filter
from PIL import Image
import imageio.v2 as imageio

FPS, GIF_W = 6, 640
BY_STATE   = True                 # color points by NM/TX instead of by radiant heat

anim   = mon.dropna(subset=["rh_per_clear_look"]).copy()
months = sorted(anim["year_month"].unique())

# ---- outage detection: months where basin-wide clear looks collapse ----
clear_by_month = anim.groupby("year_month")["n_clear_look"].sum().reindex(months)
thresh  = 0.20 * clear_by_month.rolling(13, center=True, min_periods=3).median()
outage  = set(clear_by_month.index[clear_by_month < thresh])
print("outage months (rendered as banner frames):", sorted(outage))

# ---- fixed extent + fixed color scales across every frame ----
LON    = (anim.lon.min() - 0.1, anim.lon.max() + 0.1)
LAT    = (anim.lat.min() - 0.1, anim.lat.max() + 0.1)
ASPECT = 1 / np.cos(np.deg2rad(float(anim.lat.median())))
VMAX   = float(anim["rh_per_clear_look"].quantile(0.95))

NBINS, SIGMA = 300, 2.5
def dgrid(d):
    H, _, _ = np.histogram2d(d.lon, d.lat, bins=NBINS, range=[LON, LAT],
                             weights=d["rh_per_clear_look"])
    return gaussian_filter(H.T, sigma=SIGMA)

DMAX = max(float(np.quantile(dgrid(anim[anim.year_month == m]), 0.999))
           for m in months if m not in outage)

def render(kind, tag):
    frames_dir = OUTDIR / f"frames_{tag}"
    frames_dir.mkdir(exist_ok=True)
    paths, sizes = [], None

    for m in months:
        d = anim[anim["year_month"] == m]
        fig, ax = plt.subplots(figsize=(8, 7), dpi=120)   # 960x840 -> both even, ffmpeg-safe
        fig.patch.set_facecolor("#111111")
        ax.set_facecolor("#111111")
        sm = None

        if m in outage:
            ax.text(0.5, 0.5, "SNPP sensor outage\ninsufficient clear looks",
                    ha="center", va="center", color="#dd4444", fontsize=15,
                    transform=ax.transAxes)

        elif kind == "points":
            rh = d["rh_per_clear_look"].values
            s  = 2 + 70 * np.sqrt(np.clip(rh, 0, VMAX) / VMAX)
            if BY_STATE:
                for st in ["TX", "NM"]:
                    k = (d["state"] == st).values
                    ax.scatter(d.lon[k], d.lat[k], s=s[k], c=STATE_COL[st],
                               alpha=0.85, linewidths=0, label=st)
                ax.legend(loc="lower left", frameon=False, labelcolor="0.85",
                          markerscale=1.2)
            else:
                sm = ax.scatter(d.lon, d.lat, s=s, c=rh, cmap="inferno",
                                norm=Normalize(0, VMAX), alpha=0.9, linewidths=0)

        else:  # density
            sm = ax.imshow(dgrid(d), origin="lower", extent=[*LON, *LAT],
                           cmap="inferno", norm=Normalize(0, DMAX),
                           aspect="auto", interpolation="bilinear")

        ax.set_xlim(*LON); ax.set_ylim(*LAT); ax.set_aspect(ASPECT)
        ax.plot([-103, -103], [32, LAT[1]], color="0.55", lw=0.8, ls="--")
        ax.plot([LON[0], -103], [32, 32],  color="0.55", lw=0.8, ls="--")
        ax.set_xticks([]); ax.set_yticks([])
        for sp in ax.spines.values():
            sp.set_visible(False)
        ax.text(0.02, 0.96, str(m), transform=ax.transAxes, color="w",
                fontsize=17, va="top", family="monospace")
        if sm is not None:
            cb = fig.colorbar(sm, ax=ax, fraction=0.035, pad=0.01)
            cb.set_label("RH / clear look (MW)" if kind == "points"
                         else "smoothed RH density", color="0.85")
            cb.ax.tick_params(colors="0.85")

        fp = frames_dir / f"{m}.png"
        fig.savefig(fp, facecolor=fig.get_facecolor())
        plt.close(fig)
        paths.append(fp)

    mp4 = OUTDIR / f"{tag}.mp4"
    with imageio.get_writer(mp4, fps=FPS, codec="libx264", quality=8,
                            macro_block_size=None) as w:
        for fp in paths:
            w.append_data(imageio.imread(fp))

    gif = OUTDIR / f"{tag}.gif"
    small = []
    for fp in paths:
        im = Image.open(fp).convert("RGB")
        h  = int(im.height * GIF_W / im.width)
        small.append(np.asarray(im.resize((GIF_W, h), Image.LANCZOS)))
    imageio.mimsave(gif, small, duration=1000 / FPS, loop=0)

    print(f"{tag}: {len(paths)} frames -> "
          f"{mp4.name} ({mp4.stat().st_size/1e6:.1f} MB), "
          f"{gif.name} ({gif.stat().st_size/1e6:.1f} MB)")

render("points",  "monthly_flaring_points")
render("density", "monthly_flaring_density")

outage months (rendered as banner frames): ['2022-08', '2026-07']
monthly_flaring_points: 173 frames -> monthly_flaring_points.mp4 (4.4 MB), monthly_flaring_points.gif (6.3 MB)
monthly_flaring_density: 173 frames -> monthly_flaring_density.mp4 (1.1 MB), monthly_flaring_density.gif (3.8 MB)


---
## Notes

- **Polygons vs points.** Footprints are ~280 m across and invisible at basin zoom. That layer is for
  inspecting individual sites, not for the overview; the point layers carry the basin-scale story.
- **Sizing / coloring.** Swap `RH_mean` for `Area_mean`, `N_dtct`, or the shapefile's `area`. The
  temporal layer can be recolored by *first* year, or by persistence (count of years detected), which
  is arguably more informative than last year.
- **Gif size.** ~170 frames is a lot. If the gif is unwieldy, drop `GIF_W` to 480, or subsample to every
  second month for the gif only and keep the mp4 as the full-fidelity version.
- **`BY_STATE = True`** in Product 6 turns the animation from "flaring in the Permian over time" into
  "the two arms of the experiment over time," which is the frame a viewer will actually remember. The
  cost is that radiant heat then lives only in the marker size. Set it to `False` for the heat-colored
  version.
- **Nothing here is the decoupling test.** These are descriptive artifacts. The estimand is a break in
  the flaring–production relationship — or better, in the flared *fraction* of casinghead gas, which is
  bounded on [0, 1] and tied directly to the 98% capture rule. Maps set up that question; they do not
  answer it.